# Property prompt template: three variants side by side

Use the **extract** kernel. Run cells in order. Set `PROPERTY` (and `PAPER_UID`, `SAMPLE_ID`) in the settings cell; every prompt and schema is built from the property's entry in `PROPERTY_CONFIG`, so switching property needs no other change.

1. **Per sample:** one request per sample; the model sees only the target record.
2. **Per paper:** one request per paper (split into batches of `TARGET_BATCH_SIZE` targets); the model sees all samples, writes the evidence (`paper_label`, `identifying_data`, `match_basis`) and links each value to a sample. Code keeps only strong matches.
3. **Per sample with other samples as context:** like 1, but the paper's other samples are listed to tell the samples apart.

The templates are the Tg prompts from `tg_property_prompt.ipynb` with the property-specific parts replaced by placeholders (`[[...]]`), filled from the config. Configured properties: Tg, Tm, thermal decomposition temperature, Mn, Mw and dispersity (PDI). Record fields: `property`, `method`, `subtype`, `value_raw`, `value`, `value_type`, `unit`, `conditions`, `origin`, `description` (prompt 2 adds `paper_label`, `identifying_data`, `match_basis`, `sample_id` first). Nothing is saved; the database and papers are only read.

To add a property, add one entry to `PROPERTY_CONFIG`.

## The 20 papers

ID000061162, ID000137005, ID000143254, ID000180254, ID000180835, ID000415115, ID000573272, ID000581823, ID000588793, ID000732356, ID000732480, ID000750264, ID000752848, ID000905310, ID001002844, ID001032337, ID001098286, ID001098538, ID001098910, ID001100849.

The listing cell below shows whether property presence flags the chosen property for the paper.

In [ ]:
import sys
sys.dont_write_bytecode = True
import asyncio
import json
import re
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display
from openai import AsyncOpenAI

# Find the repository from this folder or any parent.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/polymer_samples.db").is_file())

BASE_URL = "http://localhost:8000/v1"
MODEL = "qwen3.8-27b"
THINKING = True        # Set False for thinking off.
MAX_TOKENS = 20_000    # Completion tokens per request.
SHOW_REASONING = False # Print the model's reasoning trace above each answer.

## Property configurations

One entry per property. The key is the value of `property` in the output.

- `methods`: allowed methods, each with its allowed subtypes (`None` means `subtype` must be `null` for that method).
- `subtype_glossary`: optional explanations of the subtypes, shown in the prompt.
- `conditions`: allowed condition names, with an optional hint shown in the prompt.
- `exclude`: values the prompt must not extract for this property.
- `presence_label`: the label used by the property-presence classifier (`property_classification.properties_json`).
- `units`, `unit_help`, `value_examples`, `value_raw_examples`, `value_note`, `condition_examples`: the unit list and the examples shown in the value and condition fields. Temperature properties share `TEMPERATURE`; Mn, Mw and dispersity share `MOLECULAR_WEIGHT` (dispersity overrides the unit and examples).

Configured properties: Tg, Tm, thermal decomposition temperature, Mn, Mw and dispersity (PDI is the same property as dispersity).

In [274]:
# ---- shared pieces --------------------------------------------------------------------------
TEMPERATURE = {
    "units": ["°C", "K", "°F", "unspecified"],
    "unit_help": "`°C`, `K`, or `°F`, as reported; `unspecified` if no unit is stated. Never convert.",
    "value_raw_examples": '`"152.3 ± 1.2 °C"`, `"120–125 °C"`, `"> 200 °C"`',
    "value_examples": {  # value_type -> (example as written, value)
        "exact": ("152 °C", "152"), "mean_sd": ("152 ± 2 °C", "152"), "approximate": ("~150 °C", "150"),
        "range": ("120–125 °C", "125 (upper end)"), "lower_bound": ("> 200 °C", "200"),
        "upper_bound": ("< −50 °C", "−50"),
    },
    "value_note": "",
    "condition_examples": {"number": "`10`", "text": '`"N2"`, `"second_heating"`',
                           "units": '`"K/min"`, `"Hz"`, `"mL/min"`, `"°C"`'},
}

MOLECULAR_WEIGHT = {
    "units": ["g/mol", "kg/mol", "Da", "kDa", "unspecified"],
    "unit_help": "`g/mol`, `kg/mol`, `Da`, or `kDa`, as reported; `unspecified` if no unit is stated. Never convert between units.",
    "value_raw_examples": '`"12 400 g/mol"`, `"3.5 × 10^4 g/mol"`, `"25.1 ± 0.8 kg/mol"`',
    "value_examples": {
        "exact": ("12 400 g/mol", "12400"), "mean_sd": ("25.1 ± 0.8 kg/mol", "25.1"),
        "approximate": ("~5000 g/mol", "5000"), "range": ("10–12 kg/mol", "12 (upper end)"),
        "lower_bound": ("> 100 kg/mol", "100"), "upper_bound": ("< 2000 g/mol", "2000"),
    },
    "value_note": ("  A power-of-ten multiplier written with the number or in the column header is applied to `value` "
                   "(e.g., `3.5 × 10^4 g/mol` → 35000 g/mol; `12.4` in a column headed `Mn (×10³ g/mol)` → 12400 g/mol). "
                   "`value_raw` keeps the number as written. Units themselves are never converted."),
    "condition_examples": {"number": "`35`, `1.0`", "text": '`"THF"`, `"refractive index"`',
                           "units": '`"°C"`, `"mL/min"`, `"mg/mL"`'},
}

GPC_SUBTYPES = ["polystyrene_standards", "other_standards", "absolute", "unspecified"]
GPC_GLOSSARY = [
    "`polystyrene_standards`: relative to polystyrene calibration standards.",
    "`other_standards`: relative to any other calibration standards (e.g., PMMA, PEG/PEO, pullulan); name the standard in `description`.",
    "`absolute`: not relative to standards, from a light-scattering detector (e.g., MALS) or universal calibration with a viscometer detector; name which in `description`.",
]
MW_CONDITIONS = [
    ("solvent", "eluent or NMR solvent, e.g., THF, DMF with LiBr, CDCl3"), ("temperature", None),
    ("flow_rate", None), ("concentration", None), ("detector", "e.g., refractive index, MALS, UV"),
    ("column", None), ("other", None),
]
MW_EXCLUDE = [
    "theoretical or target values calculated from the monomer-to-initiator ratio or conversion (e.g., Mn,theo, Mn calcd)",
    "nominal or supplier values of purchased materials",
    "values of precursors or macroinitiators that are not the sample itself",
    "viscosity-average molecular weight (Mv) or peak molecular weight (Mp)",
]
MW_DESCRIPTION = [
    "what the value is: which polymer, block, or fraction it refers to, and the state of the material (e.g., crude product before precipitation, after purification, after degradation);",
    "the method and how the value was obtained (e.g., GPC against polystyrene standards; 1H NMR end-group analysis; MALDI-TOF);",
    "eluent or solvent, column temperature, flow rate, concentration, detector, columns, and instrument;",
    "any other measurement detail not captured in the fields.",
]

# ---- one entry per property -------------------------------------------------------------------
TG_SUBTYPES_DSC = ["onset", "midpoint", "inflection", "unspecified"]
TG_SUBTYPES_DMA = ["tan_delta_peak", "loss_modulus_peak", "storage_modulus_onset", "unspecified"]
TD_SUBTYPES_TGA = ["onset", "mass_loss_5", "mass_loss_10", "mass_loss_20", "mass_loss_50", "mass_loss_other",
                   "max_rate", "endset", "range", "unspecified"]
TD_SUBTYPES_DSC = ["onset", "peak", "endset", "unspecified"]  # DSC measures heat flow, not mass
TM_SUBTYPES = ["peak", "onset", "endset", "unspecified"]

PROPERTY_CONFIG = {
    "glass_transition_temperature": {
        **TEMPERATURE,
        "presence_label": "glass transition temperature",
        "label": "glass transition temperature (Tg)",
        "plain": "glass transition temperature",
        "short": "Tg",
        "methods": {
            "DSC": TG_SUBTYPES_DSC, "DTA": TG_SUBTYPES_DSC, "DMA": TG_SUBTYPES_DMA,
            "TMA": None, "dilatometry": None, "dielectric": None, "other": None, "unspecified": None,
        },
        "method_help": {},
        "subtype_help": "which feature of the curve was taken as Tg",
        "subtype_glossary": [],
        "separate_records": "different methods, subtypes, heating rates, scans, or frequencies",
        "exclude": ["melting, crystallization, or cold-crystallization temperatures (Tm, Tc, Tcc)"],
        "trend_example": "\"Tg increased with filler content\"",
        "not_observed_example": "no Tg detected",
        "conditions": [
            ("heating_rate", None), ("cooling_rate", None),
            ("scan", "e.g., `first_heating`, `second_heating`, `cooling`"),
            ("frequency", None), ("atmosphere", None), ("temperature_range", None),
            ("thermal_history", "treatment of the specimen just before measurement, e.g., quenching"),
            ("specimen_form", "e.g., film, powder"), ("other", None),
        ],
        "calc_examples": "Fox, Gordon–Taylor, simulation",
        "description_bullets": [
            "what the value is: which transition, phase, segment, or block (e.g., soft-segment Tg, second Tg of a blend);",
            "the method and how Tg was taken from the curve (e.g., midpoint of the heat-capacity step, tan δ peak);",
            "the scan and thermal history (e.g., second heating after cooling at 10 °C/min; annealing or quenching before the test);",
            "heating or cooling rate, frequency, strain or amplitude, temperature range, atmosphere, instrument, and specimen form, mass, or preparation for the test;",
            "any other measurement detail not captured in the fields.",
        ],
        "description_none": "Tg measured by DSC; no measurement conditions stated.",
        "description_example": "Tg of the soft polyether segment, taken as the midpoint of the heat-capacity step in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −90 to 200 °C after cooling at 10 °C/min; about 5 mg of film in an aluminium pan.",
    },
    "melting_temperature": {
        **TEMPERATURE,
        "presence_label": "melting temperature",
        "label": "melting temperature (Tm)",
        "plain": "melting temperature",
        "short": "Tm",
        "methods": {
            "DSC": TM_SUBTYPES, "DTA": TM_SUBTYPES, "optical_microscopy": None,
            "other": None, "unspecified": None,
        },
        "method_help": {"optical_microscopy": "e.g., hot-stage polarized optical microscopy"},
        "subtype_help": "which feature of the melting endotherm was taken as Tm",
        "subtype_glossary": [],
        "separate_records": "different methods, subtypes, heating rates, scans, or melting peaks (e.g., double melting)",
        "exclude": ["glass transition, crystallization, cold-crystallization, or decomposition temperatures",
                    "melting enthalpies or crystallinity"],
        "trend_example": "\"Tm decreased with comonomer content\"",
        "not_observed_example": "no melting peak observed",
        "conditions": [
            ("heating_rate", None), ("cooling_rate", None),
            ("scan", "e.g., `first_heating`, `second_heating`"),
            ("atmosphere", None), ("temperature_range", None),
            ("thermal_history", "treatment of the specimen just before measurement, e.g., annealing, isothermal crystallization"),
            ("specimen_form", "e.g., film, powder"), ("other", None),
        ],
        "calc_examples": "Hoffman–Weeks extrapolation of the equilibrium melting temperature, Gibbs–Thomson, simulation",
        "description_bullets": [
            "what the value is: which crystalline phase, block, or melting peak (e.g., low-temperature peak of a double melting endotherm, PEG block);",
            "the method and how Tm was taken from the curve (e.g., peak maximum of the melting endotherm, onset);",
            "the scan and thermal history (e.g., second heating after cooling at 10 °C/min; annealing or isothermal crystallization before the test);",
            "heating rate, temperature range, atmosphere, instrument, and specimen form, mass, or preparation for the test;",
            "any other measurement detail not captured in the fields.",
        ],
        "description_none": "Tm measured by DSC; no measurement conditions stated.",
        "description_example": "Melting temperature of the polyester hard segment, taken as the peak maximum of the melting endotherm in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −50 to 200 °C after cooling at 10 °C/min; about 5 mg in an aluminium pan.",
    },
    "thermal_decomposition_temperature": {
        **TEMPERATURE,
        "presence_label": "thermal decomposition temperature",
        "label": "thermal decomposition temperature",
        "plain": "thermal decomposition temperature",
        "short": "decomposition temperature",
        # DTG is the derivative of the TGA curve, not a separate method: a DTG peak is TGA + max_rate.
        # TGA coupled with MS or FTIR gives the same mass curve, so it is TGA.
        "methods": {"TGA": TD_SUBTYPES_TGA, "DSC": TD_SUBTYPES_DSC,
                    "other": None, "unspecified": TD_SUBTYPES_TGA},
        "method_help": {"TGA": "including TGA coupled with MS or FTIR"},
        "subtype_help": "how the decomposition temperature is defined",
        "subtype_glossary": [
            "`onset`: initial or extrapolated onset of decomposition (e.g., IDT, Ti, Tonset); for DSC, onset of the decomposition peak.",
            "`mass_loss_5`, `mass_loss_10`, `mass_loss_20`, `mass_loss_50`: temperature at that mass loss in % (e.g., T5% or Td5 → `mass_loss_5`).",
            "`mass_loss_other`: temperature at a stated mass loss not in the list (e.g., T1%, T30%; state the percentage in `description`).",
            "`max_rate`: temperature of the maximum mass-loss rate, i.e., the peak of the derivative (DTG) curve (e.g., Tmax, MDT). A reported DTG peak → method `TGA`, subtype `max_rate`.",
            "`peak`: peak temperature of the decomposition exotherm or endotherm in DSC.",
            "`endset`: end of decomposition (e.g., FDT, Tend).",
            "`range`: a stated temperature interval of a decomposition step.",
        ],
        "separate_records": "different methods, subtypes, decomposition steps, heating rates, or atmospheres",
        "exclude": [
            "residue or char yield, or weight loss in %",
            "activation energies or other kinetic parameters",
            "heat deflection or Vicat temperatures",
            "melting or glass transition temperatures (note that \"TG\" often means thermogravimetry)",
        ],
        "trend_example": "\"thermal stability increased with filler content\"",
        "not_observed_example": "no decomposition detected",
        "conditions": [
            ("heating_rate", None), ("atmosphere", "e.g., `N2`, `air`"), ("gas_flow_rate", None),
            ("temperature_range", None), ("sample_mass", None),
            ("specimen_form", "e.g., film, powder"), ("other", None),
        ],
        "calc_examples": "extrapolation to zero heating rate, molecular simulation",
        "description_bullets": [
            "what the value is: which decomposition step it belongs to (e.g., first, second, or main step) and what decomposes, if stated;",
            "the method and how the temperature is defined, with the paper's own label (e.g., temperature at 5 % mass loss, DTG peak, extrapolated onset, IDT);",
            "the atmosphere and gas flow rate (e.g., nitrogen at 50 mL/min, air);",
            "heating rate, temperature range, instrument, crucible, and specimen form, mass, or preparation for the test;",
            "any other measurement detail not captured in the fields.",
        ],
        "description_none": "Decomposition temperature measured by TGA; no measurement conditions stated.",
        "description_example": "Temperature at 5 % mass loss in the main decomposition step, from TGA under nitrogen (50 mL/min) at 10 °C/min from 30 to 600 °C; about 10 mg of powder in an alumina crucible.",
    },
    "number_average_molecular_weight": {
        **MOLECULAR_WEIGHT,
        "presence_label": "number-average molecular weight",
        "label": "number-average molecular weight (Mn)",
        "plain": "number-average molecular weight",
        "short": "Mn",
        "methods": {"GPC": GPC_SUBTYPES, "NMR": None, "other": None, "unspecified": None},
        "method_help": {"GPC": "GPC or SEC", "NMR": "end-group analysis; counts as `measured`"},
        "subtype_help": "for GPC, the calibration or detection used to obtain molar mass",
        "subtype_glossary": GPC_GLOSSARY,
        "separate_records": "different methods, calibrations, eluents, or material states (e.g., crude and purified)",
        "exclude": MW_EXCLUDE + ["weight-average molecular weight (Mw) and dispersity"],
        "trend_example": "\"Mn increased with reaction time\"",
        "not_observed_example": "not measurable (insoluble)",
        "conditions": MW_CONDITIONS,
        "calc_examples": "kinetic modelling, molecular simulation",
        "description_bullets": MW_DESCRIPTION,
        "description_none": "Mn measured by GPC; no measurement conditions stated.",
        "description_example": "Number-average molecular weight of the purified copolymer from GPC in THF at 35 °C and 1.0 mL/min with a refractive-index detector, against polystyrene standards; about 2 mg/mL, filtered through a 0.45 μm PTFE filter.",
    },
    "weight_average_molecular_weight": {
        **MOLECULAR_WEIGHT,
        "presence_label": "weight-average molecular weight",
        "label": "weight-average molecular weight (Mw)",
        "plain": "weight-average molecular weight",
        "short": "Mw",
        "methods": {"GPC": GPC_SUBTYPES, "other": None, "unspecified": None},
        "method_help": {"GPC": "GPC or SEC"},
        "subtype_help": "for GPC, the calibration or detection used to obtain molar mass",
        "subtype_glossary": GPC_GLOSSARY,
        "separate_records": "different methods, calibrations, eluents, or material states (e.g., crude and purified)",
        "exclude": MW_EXCLUDE + ["number-average molecular weight (Mn) and dispersity",
                                 "Mw values you would have to compute from Mn and dispersity"],
        "trend_example": "\"Mw increased with reaction time\"",
        "not_observed_example": "not measurable (insoluble)",
        "conditions": MW_CONDITIONS,
        "calc_examples": "kinetic modelling, molecular simulation",
        "description_bullets": MW_DESCRIPTION,
        "description_none": "Mw measured by GPC; no measurement conditions stated.",
        "description_example": "Weight-average molecular weight of the resin from static light scattering (Zimm plot) in distilled water, filtered through a 0.45 μm nylon filter, at six concentrations from 0.05 to 0.30 mg/mL and scattering angles from 30 to 135°.",
    },
    "dispersity": {
        **MOLECULAR_WEIGHT,
        "units": ["none"],
        "unit_help": "always `none` (dispersity has no unit).",
        "value_raw_examples": '`"1.18"`, `"Đ = 1.25 ± 0.03"`, `"< 1.1"`',
        "value_examples": {
            "exact": ("1.18", "1.18"), "mean_sd": ("1.25 ± 0.03", "1.25"), "approximate": ("~1.2", "1.2"),
            "range": ("1.1–1.3", "1.3 (upper end)"), "lower_bound": ("> 2", "2"), "upper_bound": ("< 1.1", "1.1"),
        },
        "value_note": "",
        "presence_label": "dispersity",
        "label": "dispersity (Đ, also called PDI or Mw/Mn)",
        "plain": "dispersity (PDI)",
        "short": "dispersity",
        "methods": {"GPC": GPC_SUBTYPES, "other": None, "unspecified": None},
        "method_help": {"GPC": "GPC or SEC"},
        "subtype_help": "for GPC, the calibration or detection used",
        "subtype_glossary": GPC_GLOSSARY,
        "separate_records": "different methods, calibrations, eluents, or material states (e.g., crude and purified)",
        "exclude": MW_EXCLUDE[1:] + ["dispersity values you would have to compute from Mn and Mw",
                                     "the dispersity of particle sizes (particle size distribution)"],
        "trend_example": "\"the distribution broadened with conversion\"",
        "not_observed_example": "not measurable (insoluble)",
        "conditions": MW_CONDITIONS,
        "calc_examples": "kinetic modelling, molecular simulation",
        "description_bullets": MW_DESCRIPTION,
        "description_none": "Dispersity measured by GPC; no measurement conditions stated.",
        "description_example": "Dispersity of the crude copolymer from GPC in THF at 40 °C and 1.0 mL/min with a refractive-index detector, against polystyrene standards.",
    },
}
VALUE_TYPES = ["exact", "mean_sd", "approximate", "range", "lower_bound", "upper_bound", "not_observed"]
CONDITION_TYPES = ["exact", "mean_sd", "approximate", "range", "lower_bound", "upper_bound"]

## Prompt templates

Placeholders in `[[...]]` are filled from `PROPERTY_CONFIG` by `build_system_prompt`. Edit the templates here; the rendered prompts can be printed below.

In [275]:
DESCRIPTION_FIELD = """- **description:** a clean, fully specified description of this measurement that can be understood without the paper or the other fields. Include everything the paper states about it, even if it is already in another field:
[[DESCRIPTION_BULLETS]]

  Do not include the value, the sample name or identifier, table or figure numbers, or comparisons with other samples. Use only what the paper states; if little is stated, say so briefly (e.g., "[[DESCRIPTION_NONE]]").

  Example: "[[DESCRIPTION_EXAMPLE]]\""""

CONDITIONS_FIELD = """- **conditions:** measurement conditions stated in the paper for this measurement. Each condition has:
  - `name`: [[CONDITION_NAMES_LONG]].
  - `type`: `exact`, `mean_sd`, `range`, `approximate`, `lower_bound`, or `upper_bound`.
  - `value`: a number when numeric (e.g., [[COND_NUMBER_EXAMPLES]]), otherwise text (e.g., [[COND_TEXT_EXAMPLES]]).
  - `unit`: e.g., [[COND_UNIT_EXAMPLES]]; `null` for non-numeric values.
  - `description`: `null`, or a short note (required when `name` is `other`).

  Include only conditions the paper states. Use an empty list if none."""

# Prompt 1: per sample.
P1_SYSTEM_TEMPLATE = """
# Task

Extract every **[[LABEL]]** reported for **one requested polymer/material sample** from the supplied paper.

Use the database record **only to identify the target sample**. All values and measurement details must come from the paper, including its text, tables, captions, footnotes, and supplementary text.

Return exactly:

`{"properties": [...]}`

If the paper reports no [[SHORT]] for this sample, return `{"properties": []}`.

## 1. Which values belong to the sample

Extract a [[SHORT]] only if the paper clearly reports it for **this exact sample**.

- Do not extract values for other samples, precursors, components, or reference materials.
- Do not extract values quoted from other publications.
- Do not extract trends without a number (e.g., [[TREND_EXAMPLE]]).
- Do not extract [[EXCLUDE]].
- Sample names can be shared by several samples, so identify the target from its whole record, especially the preparation or characterization conditions it describes, and extract a value only when the paper reports it for a sample made under those same conditions.
- If it is unclear whether a value belongs to this sample, skip it.

One record per distinct reported value:

- The same value given in both text and table → one record.
- The same [[SHORT]] measured with [[SEPARATE_RECORDS]] → separate records.

## 2. Fields

- **property:** always `[[PROPERTY]]`.
- **method:** the technique used to measure this [[SHORT]]: [[METHOD_LIST]]. General instrument descriptions in the Methods section apply to this sample's measurement.
[[SUBTYPE_FIELD]]
- **value_raw:** the value exactly as written, including ±, ranges, inequalities, and unit (e.g., [[VALUE_RAW_EXAMPLES]]).
- **value_type** and **value:** classify the reported form and set a single number:

[[VALUE_TABLE]]

[[VALUE_NOTE]]

  Use `not_observed` only when the paper explicitly states that no [[SHORT]] was detected for this sample.

- **unit:** [[UNIT_HELP]]
[[CONDITIONS_FIELD]]
- **origin:** `measured`, or `calculated` if the value comes from a model or equation (e.g., [[CALC_EXAMPLES]]).
[[DESCRIPTION_FIELD]]

## 3. Source fidelity

Never invent, infer, calculate, convert, or average values or conditions. Keep each value with its correct method and conditions. If a detail is not stated, use `unspecified`, `null`, or an empty list.

## 4. Output

Before returning, verify that:

- Every record is for the requested sample.
- No cited, plotted-only, or other-sample values are included.
- `value` follows the table above.
- `subtype` matches `method`.
- `description` is fully specified and contains no value, sample name, or table or figure number.

Return **only** the JSON object. No explanations or extra fields.
"""

P1_USER_TEMPLATE = """
Extract every [[PLAIN]] reported for the requested sample, following the system instructions.

Return exactly `{"properties": [...]}`, or `{"properties": []}` if none is reported for this sample.

Use the identification record only to identify the target material. Extract all values and measurement details exclusively from the supplied paper.

Requested record: paper_uid={paper_uid}; sample_id={sample_id}.

<IDENTIFICATION_RECORD_DATA_ONLY>
{sample_record}
</IDENTIFICATION_RECORD_DATA_ONLY>

<PAPER_MARKDOWN_DATA_ONLY>
{markdown_full_text}
</PAPER_MARKDOWN_DATA_ONLY>
"""

# Prompt 2: per paper.
P2_SYSTEM_TEMPLATE = """
# Task

Extract every **[[LABEL]]** value reported in the supplied paper and link each value to a sample from the supplied sample list.

The sample list contains every sample recorded for this paper. Use it **only to identify samples**. All values and measurement details must come from the paper, including its text, tables, captions, footnotes, and supplementary text.

Return exactly:

`{"properties": [...]}`

If the paper reports no [[SHORT]] for the target samples, return `{"properties": []}`.

## 1. Which values to report

Report every [[SHORT]] value the paper gives for a target sample, and every [[SHORT]] value that matches no sample in the list. Skip values that clearly belong to a listed sample that is not a target.

- Do not report values quoted from other publications.
- Do not read values off plots. Use only numbers stated in text, tables, or captions.
- Do not report trends without a number.
- Do not report [[EXCLUDE]].

One record per distinct reported value:

- The same value given in both text and table → one record.
- The same [[SHORT]] measured with [[SEPARATE_RECORDS]] → separate records.

## 2. Linking a value to a sample

For each value, first record what the paper says about the material, then choose the sample.

- **paper_label:** the name or label the paper uses for the material next to the value (e.g., the table row label).
- **identifying_data:** short text with the data reported together with the value that identify the material: table entry, preparation conditions, composition, molecular weight, dispersity. Use `""` if none.
- **match_basis:** how the value is tied to the chosen sample:
  - `same_entry`: the paper refers the value to the same sample entry as the record. The paper's label for the material equals the record's `identifier` or one of its `aliases` or the paper places the value in the same table entry, run, or batch as the record. No other listed sample has that label or entry, and nothing reported with the value contradicts the record.
  - `same_description`: the preparation conditions or characterization data given with the value (e.g.,
    composition, feed ratio, molecular weight, dispersity, processing) agree with the record's `description`,
    and with no other listed sample. Numbers must agree as reported; close values belong to a different sample.
  - `name_only`: only the name matches; nothing else ties the value to this sample.
  - `none`: no listed sample matches, or the data given with the value contradict every candidate.
- **sample_id:** the `sample_id` of the matched sample, or `null` when `match_basis` is `none`.

Rules:

- Several samples can share a name. A shared name alone never decides between them; compare the identifying data with each record.
- If the identifying data disagree with a sample record (e.g., a different molecular weight or different preparation conditions), the value does not belong to that sample.
- If no listed sample agrees with the identifying data, use `match_basis: none` and `sample_id: null`. This is a correct and common answer.

## 3. Value fields

- **property:** always `[[PROPERTY]]`.
- **method:** the technique used to measure this [[SHORT]]: [[METHOD_LIST]]. Table footnotes and general instrument descriptions in the Methods section count as stated.
[[SUBTYPE_FIELD]]
- **value_raw:** the value exactly as written, including ±, ranges, inequalities, and unit (e.g., [[VALUE_RAW_EXAMPLES]]).
- **value_type** and **value:** classify the reported form and set a single number:

[[VALUE_TABLE]]

[[VALUE_NOTE]]

  Use `not_observed` only when the paper explicitly states that no [[SHORT]] was detected.

- **unit:** [[UNIT_HELP]]
[[CONDITIONS_FIELD]]
- **origin:** `measured`, or `calculated` if the value comes from a model or equation (e.g., [[CALC_EXAMPLES]]).
[[DESCRIPTION_FIELD]]

## 4. Source fidelity

Never invent, infer, calculate, convert, or average values or conditions. If a detail is not stated, use `unspecified`, `null`, `""`, or an empty list.

Before returning, verify that `subtype` matches `method` and that every `description` is fully specified and contains no value, sample name, or table or figure number.

Return **only** the JSON object. No explanations or extra fields.
"""

P2_USER_TEMPLATE = """
Extract every [[PLAIN]] reported in the paper and link each value to a sample from the sample list, following the system instructions.

Return exactly `{"properties": [...]}`.

paper_uid={paper_uid}
Target samples: {target_sample_ids}

<SAMPLE_LIST_DATA_ONLY>
{sample_list}
</SAMPLE_LIST_DATA_ONLY>

<PAPER_MARKDOWN_DATA_ONLY>
{markdown_full_text}
</PAPER_MARKDOWN_DATA_ONLY>
"""

# Prompt 3: per sample, other samples as context.
P3_SYSTEM_TEMPLATE = P1_SYSTEM_TEMPLATE.replace(
    "Use the database record **only to identify the target sample**.",
    "Use the target record and the list of the paper's other samples **only to identify the target sample**.",
).replace(
    "- If it is unclear whether a value belongs to this sample, skip it.",
    "- The paper's other samples are listed only so you can tell the samples apart. Compare the target record with them, and extract a value only if the paper reports it for the target and not for one of the other listed samples or for a material that is not in the list. Never extract values for the other samples.\n"
    "- If it is unclear whether a value belongs to this sample, skip it.",
)
assert P3_SYSTEM_TEMPLATE != P1_SYSTEM_TEMPLATE and "other samples are listed only" in P3_SYSTEM_TEMPLATE

P3_USER_TEMPLATE = """
Extract every [[PLAIN]] reported for the requested sample, following the system instructions.

Return exactly `{"properties": [...]}`, or `{"properties": []}` if none is reported for this sample.

Use the target record and the other samples only to identify the target material. Extract all values and measurement details exclusively from the supplied paper.

Requested record: paper_uid={paper_uid}; sample_id={sample_id}.

<IDENTIFICATION_RECORD_DATA_ONLY>
{sample_record}
</IDENTIFICATION_RECORD_DATA_ONLY>

Other samples in this paper, for telling samples apart only (do not extract values for them):

<OTHER_SAMPLES_DATA_ONLY>
{other_samples}
</OTHER_SAMPLES_DATA_ONLY>

<PAPER_MARKDOWN_DATA_ONLY>
{markdown_full_text}
</PAPER_MARKDOWN_DATA_ONLY>
"""

TEMPLATES = {
    "per_sample": (P1_SYSTEM_TEMPLATE, P1_USER_TEMPLATE),
    "per_paper": (P2_SYSTEM_TEMPLATE, P2_USER_TEMPLATE),
    "with_context": (P3_SYSTEM_TEMPLATE, P3_USER_TEMPLATE),
}

## Build prompts and schemas from the config

`build_system_prompt(prop, mode)` fills a template; `build_user_prompt(prop, mode)` gives the user template (its `{...}` fields are filled per request). `make_response_format(prop)` is the schema for prompts 1 and 3; `make_paper_response_format(prop, sample_ids)` is the schema for prompt 2 (sample IDs limited to the paper). All enums come from the config. The check after the cell confirms that no placeholder is left in any rendered prompt.

In [276]:
def _code_list(items, last="or"):
    items = [f"`{i}`" for i in items]
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + f", {last} " + items[-1]


def _method_list(cfg):
    names = [m for m in cfg["methods"] if m != "unspecified"]
    parts = [f"`{m}`" + (f" ({cfg['method_help'][m]})" if m in cfg["method_help"] else "") for m in names]
    text = ", ".join(parts)
    if "unspecified" in cfg["methods"]:
        text += ", or `unspecified` (if not stated)"
    return text


def _subtype_field(cfg):
    lines = [f"- **subtype:** {cfg['subtype_help']}. Use only if the paper states it; never infer it."]
    lines += [f"  - {g}" for g in cfg["subtype_glossary"]]
    groups = {}
    for method, subtypes in cfg["methods"].items():
        groups.setdefault(tuple(subtypes) if subtypes else None, []).append(method)
    single = len(groups) == 1
    for subtypes, methods in groups.items():
        who = "Any method" if single else " or ".join(methods) if len(methods) <= 2 else ", ".join(methods[:-1]) + ", or " + methods[-1]
        lines.append(f"  - {who}: " + ("`null`." if subtypes is None else _code_list(subtypes) + "."))
    lines.append("  - A stated subtype not in these lists → `unspecified`, with the detail in `description`.")
    return "\n".join(lines)


def _condition_names(cfg, with_help):
    parts = []
    for name, hint in cfg["conditions"]:
        parts.append(f"`{name}`" + (f" ({hint})" if with_help and hint else ""))
    return ", ".join(parts[:-1]) + ", or " + parts[-1]


def _exclude(cfg):
    items = cfg["exclude"]
    return items[0] if len(items) == 1 else "; ".join(items[:-1]) + "; or " + items[-1]


def _value_table(cfg):
    rows = ["| value_type | Example | value |", "|---|---|---|"]
    for vtype in VALUE_TYPES[:-1]:
        example, value = cfg["value_examples"][vtype]
        rows.append(f"| `{vtype}` | `{example}` | {value} |")
    rows.append(f"| `not_observed` | `{cfg['not_observed_example']}` | `null` |")
    return "\n".join(rows)


def _fill(text, values):
    for key, value in values.items():
        text = text.replace(f"[[{key}]]", value)
    return text


def build_system_prompt(prop, mode):
    cfg = PROPERTY_CONFIG[prop]
    blocks = {
        "VALUE_TABLE": _value_table(cfg),
        "CONDITIONS_FIELD": CONDITIONS_FIELD,
        "DESCRIPTION_FIELD": DESCRIPTION_FIELD,
    }
    values = {
        "LABEL": cfg["label"], "PLAIN": cfg["plain"], "SHORT": cfg["short"], "PROPERTY": prop,
        "METHOD_LIST": _method_list(cfg),
        "SUBTYPE_FIELD": _subtype_field(cfg),
        "SEPARATE_RECORDS": cfg["separate_records"],
        "EXCLUDE": _exclude(cfg),
        "TREND_EXAMPLE": cfg["trend_example"],
        "NOT_OBSERVED_EXAMPLE": cfg["not_observed_example"],
        "CONDITION_NAMES_LONG": _condition_names(cfg, with_help=True),
        "CONDITION_NAMES_SHORT": _condition_names(cfg, with_help=False),
        "CALC_EXAMPLES": cfg["calc_examples"],
        "DESCRIPTION_BULLETS": "\n".join(f"  - {b}" for b in cfg["description_bullets"]),
        "DESCRIPTION_NONE": cfg["description_none"],
        "DESCRIPTION_EXAMPLE": cfg["description_example"],
        "VALUE_RAW_EXAMPLES": cfg["value_raw_examples"],
        "VALUE_NOTE": cfg["value_note"],
        "UNIT_HELP": cfg["unit_help"],
        "COND_NUMBER_EXAMPLES": cfg["condition_examples"]["number"],
        "COND_TEXT_EXAMPLES": cfg["condition_examples"]["text"],
        "COND_UNIT_EXAMPLES": cfg["condition_examples"]["units"],
    }
    text = _fill(TEMPLATES[mode][0], blocks)  # insert the shared blocks first, then fill everything
    text = _fill(text, values)
    text = re.sub(r"\n{3,}", "\n\n", text)  # an empty block (e.g., no value note) leaves no gap
    return text.strip() + "\n"


def build_user_prompt(prop, mode):
    cfg = PROPERTY_CONFIG[prop]
    return _fill(TEMPLATES[mode][1], {"PLAIN": cfg["plain"]})


def _record_properties(prop):
    cfg = PROPERTY_CONFIG[prop]
    subtypes = sorted({s for subs in cfg["methods"].values() if subs for s in subs})
    has_null = any(subs is None for subs in cfg["methods"].values())
    return {
        "property": {"type": "string", "enum": [prop]},
        "method": {"type": "string", "enum": list(cfg["methods"])},
        "subtype": ({"type": ["string", "null"], "enum": subtypes + [None]} if has_null
                    else {"type": "string", "enum": subtypes}),
        "value_raw": {"type": "string"},
        "value": {"type": ["number", "null"]},
        "value_type": {"type": "string", "enum": VALUE_TYPES},
        "unit": {"type": "string", "enum": cfg["units"]},
        "conditions": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["name", "type", "value", "unit", "description"],
                "properties": {
                    "name": {"type": "string", "enum": [n for n, _ in cfg["conditions"]]},
                    "type": {"type": "string", "enum": CONDITION_TYPES},
                    "value": {"type": ["number", "string"]},
                    "unit": {"type": ["string", "null"]},
                    "description": {"type": ["string", "null"]},
                },
            },
        },
        "origin": {"type": "string", "enum": ["measured", "calculated"]},
        "description": {"type": "string"},
    }


RECORD_FIELDS = ["property", "method", "subtype", "value_raw", "value", "value_type",
                 "unit", "conditions", "origin", "description"]


def _wrap(name, item_properties, required):
    return {
        "type": "json_schema",
        "json_schema": {
            "name": name,
            "strict": True,
            "schema": {
                "type": "object",
                "additionalProperties": False,
                "required": ["properties"],
                "properties": {"properties": {"type": "array", "items": {
                    "type": "object", "additionalProperties": False,
                    "required": required, "properties": item_properties,
                }}},
            },
        },
    }


def make_response_format(prop):
    # Prompts 1 and 3.
    return _wrap(f"sample_{prop}", _record_properties(prop), RECORD_FIELDS)


def make_paper_response_format(prop, sample_ids):
    # Prompt 2: linking fields first, so the model writes the evidence before it picks the sample.
    linking = {
        "paper_label": {"type": "string"},
        "value_raw": {"type": "string"},
        "identifying_data": {"type": "string"},
        "match_basis": {"type": "string",
                        "enum": ["same_entry", "same_description", "name_only", "none"]},
        "sample_id": {"type": ["integer", "null"], "enum": [int(i) for i in sample_ids] + [None]},
    }
    record = _record_properties(prop)
    properties = {**linking, **{k: v for k, v in record.items() if k != "value_raw"}}
    required = ["paper_label", "value_raw", "identifying_data", "match_basis", "sample_id"] + \
               [f for f in RECORD_FIELDS if f != "value_raw"]
    return _wrap(f"paper_{prop}", properties, required)


def check_record(record, prop):
    # Rules strict JSON schema cannot express.
    cfg = PROPERTY_CONFIG[prop]
    problems = []
    if (record.get("value") is None) != (record.get("value_type") == "not_observed"):
        problems.append("value must be null exactly when value_type is not_observed")
    method, subtype = record.get("method"), record.get("subtype")
    if method not in cfg["methods"]:
        problems.append(f"method {method!r} not allowed")
    else:
        allowed = cfg["methods"][method]
        if allowed is None and subtype is not None:
            problems.append(f"subtype must be null for method {method}")
        if allowed is not None and subtype not in allowed:
            problems.append(f"subtype {subtype!r} not allowed for {method}")
    return "; ".join(problems)


# Every template must render with no placeholder left, for every property.
for _prop in PROPERTY_CONFIG:
    for _mode in TEMPLATES:
        for _text in (build_system_prompt(_prop, _mode), build_user_prompt(_prop, _mode)):
            left = re.findall(r"\[\[[A-Z_]+\]\]", _text)
            assert not left, f"{_prop}/{_mode}: unfilled {left}"
print("Properties:", ", ".join(PROPERTY_CONFIG))
print("All templates render for every property.")

Properties: glass_transition_temperature, melting_temperature, thermal_decomposition_temperature, number_average_molecular_weight, weight_average_molecular_weight, dispersity
All templates render for every property.


### Read a rendered prompt

Set `SHOW_PROPERTY` and `SHOW_MODE` (`per_sample`, `per_paper`, `with_context`) to print the exact system prompt that will be sent.

In [277]:
print(P3_SYSTEM_TEMPLATE)


# Task

Extract every **[[LABEL]]** reported for **one requested polymer/material sample** from the supplied paper.

Use the target record and the list of the paper's other samples **only to identify the target sample**. All values and measurement details must come from the paper, including its text, tables, captions, footnotes, and supplementary text.

Return exactly:

`{"properties": [...]}`

If the paper reports no [[SHORT]] for this sample, return `{"properties": []}`.

## 1. Which values belong to the sample

Extract a [[SHORT]] only if the paper clearly reports it for **this exact sample**.

- Do not extract values for other samples, precursors, components, or reference materials.
- Do not extract values quoted from other publications.
- Do not extract trends without a number (e.g., [[TREND_EXAMPLE]]).
- Do not extract [[EXCLUDE]].
- Sample names can be shared by several samples, so identify the target from its whole record, especially the preparation or characterization conditi

In [278]:
SHOW_PROPERTY = "thermal_decomposition_temperature"
SHOW_MODE = "with_context"
print(build_system_prompt(SHOW_PROPERTY, SHOW_MODE))

# Task

Extract every **thermal decomposition temperature** reported for **one requested polymer/material sample** from the supplied paper.

Use the target record and the list of the paper's other samples **only to identify the target sample**. All values and measurement details must come from the paper, including its text, tables, captions, footnotes, and supplementary text.

Return exactly:

`{"properties": [...]}`

If the paper reports no decomposition temperature for this sample, return `{"properties": []}`.

## 1. Which values belong to the sample

Extract a decomposition temperature only if the paper clearly reports it for **this exact sample**.

- Do not extract values for other samples, precursors, components, or reference materials.
- Do not extract values quoted from other publications.
- Do not extract trends without a number (e.g., "thermal stability increased with filler content").
- Do not extract residue or char yield, or weight loss in %; activation energies or other ki

## Helpers

Database access, response parsing and display helpers.

In [ ]:
def format_conditions(conditions):
    parts = []
    for condition in conditions or []:
        text = f"{condition.get('name')}={condition.get('value')}"
        if condition.get("unit"):
            text += f" {condition['unit']}"
        if condition.get("type") not in (None, "exact"):
            text += f" ({condition['type']})"
        parts.append(text)
    return "; ".join(parts)


async def ask_qwen(client, messages, response_format):
    return await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
        max_tokens=MAX_TOKENS,
        extra_body={"chat_template_kwargs": {"enable_thinking": THINKING}},
        response_format=response_format,
    )


def get_reasoning(response):
    # vLLM returns the thinking trace as `reasoning_content` (older) or `reasoning` (newer).
    if isinstance(response, Exception):
        return ""
    message = response.choices[0].message
    reasoning = getattr(message, "reasoning_content", None) or getattr(message, "reasoning", None)
    if not reasoning and getattr(message, "model_extra", None):
        reasoning = message.model_extra.get("reasoning_content") or message.model_extra.get("reasoning")
    if not reasoning:
        match = re.search(r"<think>(.*?)</think>", message.content or "", flags=re.S)
        reasoning = match.group(1) if match else ""
    return (reasoning or "").strip()


def print_reasoning(response):
    if not SHOW_REASONING:
        return
    print("--- Reasoning ---")
    print(get_reasoning(response) or "[No reasoning returned; is THINKING on?]")
    print("--- Answer ---")


def parse_response(response):
    # Returns (parsed JSON or None, note); note is empty when everything went fine.
    if isinstance(response, Exception):
        return None, f"request failed: {response!r}"
    notes = []
    if response.choices[0].finish_reason != "stop":
        notes.append(f"Response may be incomplete. Finish reason: {response.choices[0].finish_reason}")
    answer = response.choices[0].message.content or ""
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()
    try:
        return json.loads(answer), "\n".join(notes)
    except json.JSONDecodeError:
        notes.append(answer or "[No answer content returned]")
        return None, "\n".join(notes)


def read_db(query, params):
    connection = sqlite3.connect(f"file:{ROOT / 'data/polymer_samples.db'}?mode=ro", uri=True)
    try:
        return pd.read_sql_query(query, connection, params=params)
    finally:
        connection.close()


def get_samples(paper_uid):
    return read_db(
        "SELECT s.paper_uid, p.canonical_source AS source, s.sample_id, "
        "s.name, s.identifier, s.aliases, s.description "
        "FROM samples s JOIN papers p USING (paper_uid) "
        "WHERE s.paper_uid = ? ORDER BY s.sample_id",
        (paper_uid,),
    )


def get_paper_title(paper_uid):
    rows = read_db("SELECT title FROM papers WHERE paper_uid = ?", (paper_uid,))
    return rows.title.iloc[0] if not rows.empty and rows.title.iloc[0] else "[no title in database]"


def get_property_presence(paper_uid):
    rows = read_db("SELECT status, properties_json FROM property_classification WHERE paper_uid = ?",
                   (paper_uid,))
    if rows.empty:
        return "not classified", []
    if rows.status.iloc[0] != "success":
        return rows.status.iloc[0], []
    return "success", [name.strip().lower() for name in json.loads(rows.properties_json.iloc[0])]


def sample_json(r):
    return json.dumps({
        "sample_id": int(r.sample_id), "name": r.name, "identifier": r.identifier,
        "aliases": json.loads(r.aliases), "description": r.description,
    }, ensure_ascii=False)


def record_row(r):
    # Flattened record for the result tables.
    return {
        "method": r.get("method"), "subtype": r.get("subtype"),
        "value_raw": r.get("value_raw"), "value": r.get("value"), "value_type": r.get("value_type"),
        "unit": r.get("unit"), "origin": r.get("origin"),
        "conditions": format_conditions(r.get("conditions")),
        "description": r.get("description"),
    }

def print_records(title, records, extra=None):
    # Every record as indented JSON with all fields.
    print(f"\n--- {title}: {len(records)} record(s) ---")
    if not records:
        print("(no records)")
    for n, r in enumerate(records, 1):
        print(f"\n[{n}]" + (f" {extra(r)}" if extra else ""))
        print(json.dumps(r, ensure_ascii=False, indent=2))


def sample_title(sample_id):
    r = names.loc[sample_id]
    return f"{sample_id}: {r['name']} | {r['identifier']}"


def records_of(response):
    parsed, note = parse_response(response)
    records = parsed.get("properties") if isinstance(parsed, dict) else None
    return records, note

## Choose the property, a paper and a sample

Set `PROPERTY` to a key of `PROPERTY_CONFIG`, set `PAPER_UID`, run the listing cell, then set `SAMPLE_ID` to one of the listed IDs. `RUN_ALL_SAMPLES = True` runs prompts 1 and 3 for every sample of the paper (one request per sample) so all three prompts cover the same samples. The send cells build their prompts from `PROPERTY` each time they run.

In [260]:
# Paper UID	Samples	Title (short)
# ID000415115	6	Chemically modified lignin in PCL nanofibers
# ID000573272	5	Glycolysis of waste PET + PTMG copolymers
# ID000588793	13	PEB-b-PTMO multiblock copolymers, elasticity
# ID000732480	6	PLA with thionoester linkages
# ID000752848	5	Cellulose nanocrystal trans-crystallisation
# ID001032337	6	Biobased thermoplastic elastomers

# Two more have five of the six:

# ID000581823 (PS-b-P4VP): everything except Tm.
# ID001002844 (poly(ethylene phosphate) block copolymers): everything except Td.

In [261]:
PROPERTY = "melting_temperature"   # one of: glass_transition_temperature, melting_temperature, thermal_decomposition_temperature,
                                                    #         number_average_molecular_weight, weight_average_molecular_weight, dispersity
PAPER_UID = "ID001032337"
SAMPLE_ID = None          # set to a sample_id from the list below
RUN_ALL_SAMPLES = True   # True: prompts 1 and 3 run every sample of the paper
MAX_CONCURRENT = 4        # parallel requests for prompts 1 and 3
TARGET_BATCH_SIZE = 15    # prompt 2: target samples per request

In [262]:
if PROPERTY not in PROPERTY_CONFIG:
    raise ValueError(f"PROPERTY must be one of {list(PROPERTY_CONFIG)}")
CFG = PROPERTY_CONFIG[PROPERTY]

paper_samples = get_samples(PAPER_UID)
if paper_samples.empty:
    raise ValueError(f"No samples for {PAPER_UID}.")
md_path = ROOT / "data/fulltext" / paper_samples.source.iloc[0] / PAPER_UID / f"{PAPER_UID}.md"
paper_markdown = md_path.read_text(encoding="utf-8")
if not paper_markdown.strip():
    raise ValueError(f"The paper Markdown is empty: {md_path}")
names = paper_samples.set_index("sample_id")

status, present = get_property_presence(PAPER_UID)
print(f"Property: {PROPERTY}")
print(f"{PAPER_UID}: {get_paper_title(PAPER_UID)}")
print(f"{len(paper_samples)} samples; full text {len(paper_markdown):,} characters")
print(f"Property presence ({status}): {', '.join(present) or 'none'}")
print(f"{CFG['presence_label']!r} flagged for this paper:", CFG["presence_label"] in present)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(paper_samples[["sample_id", "name", "identifier", "aliases", "description"]])

if RUN_ALL_SAMPLES:
    run_ids = [int(i) for i in paper_samples.sample_id]
else:
    if SAMPLE_ID is None or int(SAMPLE_ID) not in set(paper_samples.sample_id.astype(int)):
        raise ValueError("Set SAMPLE_ID to one of the sample_ids above (or RUN_ALL_SAMPLES = True).")
    run_ids = [int(SAMPLE_ID)]
print(f"Prompts 1 and 3 will run for: {run_ids}")

Property: melting_temperature
ID001032337: Biobased Thermoplastic Elastomers: Structure-Property Relationship of Poly(hexamethylene 2,5-furanodicarboxylate)-Block-Poly(tetrahydrofuran) Copolymers Prepared by Melt Polycondensation.
6 samples; full text 49,647 characters
Property presence (success): number-average molecular weight, weight-average molecular weight, dispersity, degree of polymerization, glass transition temperature, melting temperature, thermal decomposition temperature, young's modulus, tensile strength, elongation at break, storage modulus, degree of crystallinity, d-spacing
'melting temperature' flagged for this paper: True


,sample_id,name,identifier,aliases,description
0,1555566,PHF,PHF,"[""neat PHF"",""homopolymer PHF""]","Poly(hexamethylene 2,5-furanodicarboxylate) homopolymer synthesized via a two-stage melt polycondensation process. The first stage involved transesterification of dimethyl 2,5-furanodicarboxylate (DMFDC) and 1,6-hexanediol (HDO) at a 1:1.5 molar ratio with tetrabutyl orthotitanate catalyst at 160–180 °C for approximately 2 h under nitrogen, distilling off methanol. The temperature was then raised to 210 °C, and polycondensation was carried out at 240 °C under vacuum (<25 Pa) for 4–6 h without adding poly(tetrahydrofuran) (pTHF). Distinguished from the copolymers by the absence of the flexible pTHF segment and the higher polycondensation temperature (240 °C)."
1,1555567,PHF-b-F-pTHF,PHF-b-F-pTHF 75/25,"[""PHF-b-F-pTHF copolymer (75/25)""]","Poly(hexamethylene 2,5-furanodicarboxylate)-block-poly(tetrahydrofuran) copolymer with a 75/25 weight ratio of rigid PHF to flexible F-pTHF segments. Synthesized via transesterification of DMFDC and HDO (1:1.5 molar ratio) with Ti(OBu)4 at 160–180 °C for ~2 h, followed by addition of polyTHF®1000 (Mn 1000 g/mol), Irganox 1010 stabilizer, and a second portion of catalyst. Polycondensation was conducted at 235 °C under vacuum (<25 Pa) for 4–6 h. Distinguished by its 25 wt.% F-pTHF content."
2,1555568,PHF-b-F-pTHF,PHF-b-F-pTHF 65/35,"[""PHF-b-F-pTHF copolymer (65/35)""]","Poly(hexamethylene 2,5-furanodicarboxylate)-block-poly(tetrahydrofuran) copolymer with a 65/35 weight ratio of rigid PHF to flexible F-pTHF segments. Prepared using the same two-stage melt polycondensation procedure as the other samples, but with a feed ratio corresponding to 35 wt.% F-pTHF content. Polycondensation was carried out at 235 °C under vacuum (<25 Pa) for 4–6 h. Distinguished by its 35 wt.% F-pTHF content."
3,1555569,PHF-b-F-pTHF,PHF-b-F-pTHF 50/50,"[""PHF-b-F-pTHF copolymer (50/50)""]","Poly(hexamethylene 2,5-furanodicarboxylate)-block-poly(tetrahydrofuran) copolymer with a 50/50 weight ratio of rigid PHF to flexible F-pTHF segments. Synthesized via the same two-stage melt polycondensation method, adjusted to yield 50 wt.% F-pTHF content. Polycondensation was performed at 235 °C under vacuum (<25 Pa) for 4–6 h. Distinguished by its 50 wt.% F-pTHF content."
4,1555570,PHF-b-F-pTHF,PHF-b-F-pTHF 35/65,"[""PHF-b-F-pTHF copolymer (35/65)""]","Poly(hexamethylene 2,5-furanodicarboxylate)-block-poly(tetrahydrofuran) copolymer with a 35/65 weight ratio of rigid PHF to flexible F-pTHF segments. Prepared using the identical two-stage synthesis protocol, formulated to contain 65 wt.% F-pTHF. Polycondensation was conducted at 235 °C under vacuum (<25 Pa) for 4–6 h. Distinguished by its 65 wt.% F-pTHF content."
5,1555571,PHF-b-F-pTHF,PHF-b-F-pTHF 25/75,"[""PHF-b-F-pTHF copolymer (25/75)""]","Poly(hexamethylene 2,5-furanodicarboxylate)-block-poly(tetrahydrofuran) copolymer with a 25/75 weight ratio of rigid PHF to flexible F-pTHF segments. Synthesized via the same two-stage melt polycondensation route, adjusted to yield 75 wt.% F-pTHF content. Polycondensation was carried out at 235 °C under vacuum (<25 Pa) for 4–6 h. Distinguished by its 75 wt.% F-pTHF content and investigated as a thermoplastic shape memory polymer."


Prompts 1 and 3 will run for: [1555566, 1555567, 1555568, 1555569, 1555570, 1555571]


## Send prompt 1 (per sample)

One request per sample in `run_ids`. Records are in `p1_records`; reasoning in `p1_reasoning`.

In [263]:
p1_property = PROPERTY
p1_system = build_system_prompt(PROPERTY, "per_sample")
p1_user = build_user_prompt(PROPERTY, "per_sample")
p1_format = make_response_format(PROPERTY)


def build_p1_messages(sample_id):
    t = paper_samples.loc[paper_samples.sample_id == sample_id].iloc[0]
    values = {
        "paper_uid": PAPER_UID,
        "sample_id": str(sample_id),
        "sample_record": json.dumps({k: t[k] for k in ("name", "identifier", "aliases", "description")},
                                    ensure_ascii=False),
        "markdown_full_text": paper_markdown,
    }
    user = re.sub(r"\{(paper_uid|sample_id|sample_record|markdown_full_text)\}",
                  lambda m: values[m.group(1)], p1_user)
    return [{"role": "system", "content": p1_system}, {"role": "user", "content": user}]


semaphore = asyncio.Semaphore(MAX_CONCURRENT)
async with AsyncOpenAI(base_url=BASE_URL, api_key="none", timeout=1200, max_retries=0) as client:
    async def run_one(sample_id):
        async with semaphore:
            return await ask_qwen(client, build_p1_messages(sample_id), p1_format)
    p1_responses = dict(zip(run_ids, await asyncio.gather(*(run_one(i) for i in run_ids),
                                                          return_exceptions=True)))
p1_reasoning = {i: get_reasoning(r) for i, r in p1_responses.items()}

rows = []
print(f"Property: {PROPERTY}")
for sample_id, response in p1_responses.items():
    print(f"\n=== {sample_id}: {names['name'][sample_id]} | {names['identifier'][sample_id]} ===")
    print_reasoning(response)
    parsed, note = parse_response(response)
    records = parsed.get("properties") if isinstance(parsed, dict) else None
    if records is None:
        print(note or "[No parsable answer]")
        continue
    if not records:
        print("No values reported for this sample.")
    for r in records:
        rows.append({"sample_id": sample_id, **record_row(r), "check": check_record(r, PROPERTY)})
    if note:
        print(note)

p1_records = pd.DataFrame(rows)
if not p1_records.empty:
    dup = p1_records.groupby(["value_raw", "method", "conditions"]).sample_id.transform("nunique") > 1
    p1_records.loc[dup, "check"] = (p1_records.loc[dup, "check"] + "; assigned to several samples").str.strip("; ")
    print(f"\nPrompt 1 values ({len(p1_records)}):")
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(p1_records)

Property: melting_temperature

=== 1555566: PHF | PHF ===

=== 1555567: PHF-b-F-pTHF | PHF-b-F-pTHF 75/25 ===
No values reported for this sample.

=== 1555568: PHF-b-F-pTHF | PHF-b-F-pTHF 65/35 ===
No values reported for this sample.

=== 1555569: PHF-b-F-pTHF | PHF-b-F-pTHF 50/50 ===
No values reported for this sample.

=== 1555570: PHF-b-F-pTHF | PHF-b-F-pTHF 35/65 ===

=== 1555571: PHF-b-F-pTHF | PHF-b-F-pTHF 25/75 ===

Prompt 1 values (4):


,sample_id,method,subtype,value_raw,value,value_type,unit,origin,conditions,description,check
0,1555566,DSC,peak,151.3 °C,151.3,exact,°C,measured,scan=second_heating; heating_rate=10 °C/min; cooling_rate=10 °C/min; atmosphere=N2; temperature_range=-120 to 225 °C (range),"Melting temperature taken as the maximum of the melting endotherm in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −120 to 225 °C after cooling at 10 °C/min.",
1,1555570,DSC,peak,−7.1 °C,-7.1,exact,°C,measured,scan=second_heating; heating_rate=10 °C/min; cooling_rate=10 °C/min; atmosphere=N2; temperature_range=-120 to 225 °C (range),"Melting temperature of the pTHF crystalline phase, taken as the maximum of the melting endotherm in the second DSC heating scan of a heating-cooling-heating cycle, recorded under nitrogen at 10 °C/min heating and cooling rates over a temperature range of −120 to 225 °C; no specimen form or mass stated.",
2,1555571,DSC,peak,−2.1 °C,-2.1,exact,°C,measured,heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=−120 to 225 °C (range),"Melting temperature of the pTHF crystalline phase, taken as the peak maximum of the melting endotherm in the second DSC heating scan on a Mettler Toledo DSC1 calibrated with indium and n-octane, recorded at 10 °C/min under nitrogen from −120 to 225 °C after cooling at 10 °C/min; no specimen form or mass stated.",
3,1555571,DSC,peak,−2.8 °C,-2.8,exact,°C,measured,heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=−120 to 225 °C (range),"Low-temperature melting temperature of the pTHF crystalline phase, taken as the peak maximum of the melting endotherm in the second DSC heating scan on a Mettler Toledo DSC1 calibrated with indium and n-octane, recorded at 10 °C/min under nitrogen from −120 to 225 °C after cooling at 10 °C/min; no specimen form or mass stated.",


## Send prompt 2 (per paper)

Needs only `PAPER_UID`. Every request lists all samples; papers with more than `TARGET_BATCH_SIZE` samples are split into requests that differ only in their targets.

Code decides which values to keep: `sample_id` set and `match_basis` is `same_entry`, `same_conditions` or `same_characterization`; `name_only` is kept only when no other sample in the paper has the same name. All values, kept or not, are in `p2_records`; reasoning traces are in `p2_reasoning`.

In [264]:
STRONG_MATCHES = {"same_entry", "same_description"}  # name_only is never kept; none has no sample

p2_property = PROPERTY
p2_system = build_system_prompt(PROPERTY, "per_paper")
p2_user = build_user_prompt(PROPERTY, "per_paper")

all_ids = [int(i) for i in paper_samples.sample_id]
sample_list = "\n".join(sample_json(r) for r in paper_samples.itertuples())
batches = [all_ids[i:i + TARGET_BATCH_SIZE] for i in range(0, len(all_ids), TARGET_BATCH_SIZE)]
p2_format = make_paper_response_format(PROPERTY, all_ids)


def build_p2_messages(targets):
    values = {
        "paper_uid": PAPER_UID,
        "target_sample_ids": "all samples in the list" if len(batches) == 1 else ", ".join(map(str, targets)),
        "sample_list": sample_list,
        "markdown_full_text": paper_markdown,
    }
    user = re.sub(r"\{(paper_uid|target_sample_ids|sample_list|markdown_full_text)\}",
                  lambda m: values[m.group(1)], p2_user)
    return [{"role": "system", "content": p2_system}, {"role": "user", "content": user}]


print(f"Property: {PROPERTY}")
print(f"{PAPER_UID}: {get_paper_title(PAPER_UID)}")
print(f"{len(all_ids)} samples, {len(batches)} request(s)")

async with AsyncOpenAI(base_url=BASE_URL, api_key="none", timeout=1200, max_retries=0) as client:
    p2_responses = await asyncio.gather(
        *(ask_qwen(client, build_p2_messages(targets), p2_format) for targets in batches),
        return_exceptions=True,
    )
p2_reasoning = [get_reasoning(r) for r in p2_responses]

name_counts = paper_samples.name.str.strip().str.lower().value_counts()


def p2_kept(r):
    # Keep only links the model marks as same_entry or same_description, with a sample_id.
    return r.get("sample_id") is not None and r.get("match_basis") in STRONG_MATCHES


rows, seen_unmatched = [], set()
for b, (targets, response) in enumerate(zip(batches, p2_responses), 1):
    print(f"\n=== Request {b}/{len(batches)} ===")
    print_reasoning(response)
    parsed, note = parse_response(response)
    if note:
        print(note)
    for r in (parsed or {}).get("properties", []):
        sid = r.get("sample_id")
        if sid is not None and sid not in targets:
            continue  # belongs to another batch's targets
        if sid is None:
            key = (r.get("paper_label"), r.get("value_raw"), r.get("method"))
            if key in seen_unmatched:
                continue
            seen_unmatched.add(key)
        problems = [p for p in [check_record(r, PROPERTY)] if p]
        if (r.get("match_basis") == "none") != (sid is None):
            problems.append("match_basis none must go with sample_id null")
        rows.append({
            "kept": p2_kept(r),
            "sample_id": sid,
            "sample": f"{names['name'].get(sid, '')} | {names['identifier'].get(sid, '')}" if sid is not None else "",
            "paper_label": r.get("paper_label"),
            "identifying_data": r.get("identifying_data"),
            "match_basis": r.get("match_basis"),
            **record_row(r),
            "check": "; ".join(problems),
        })

p2_records = pd.DataFrame(rows)
if p2_records.empty:
    print("\nNo values returned for this paper.")
else:
    kept_rows = p2_records[p2_records.kept]
    dup = kept_rows.groupby(["value_raw", "method", "conditions"]).sample_id.transform("nunique") > 1
    idx = kept_rows.index[dup.to_numpy()]
    p2_records.loc[idx, "check"] = (p2_records.loc[idx, "check"] + "; assigned to several samples").str.strip("; ")
    print(f"\nAll values ({len(p2_records)}; kept {int(p2_records.kept.sum())}):")
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(p2_records)

    summary = paper_samples[["sample_id", "name", "identifier"]].copy()
    kept_values = (p2_records[p2_records.kept].groupby("sample_id").value_raw.apply(lambda v: "; ".join(v)))
    summary["kept"] = summary.sample_id.map(kept_values).fillna("")
    print("\nKept values per sample:")
    with pd.option_context("display.max_rows", None):
        display(summary)

Property: melting_temperature
ID001032337: Biobased Thermoplastic Elastomers: Structure-Property Relationship of Poly(hexamethylene 2,5-furanodicarboxylate)-Block-Poly(tetrahydrofuran) Copolymers Prepared by Melt Polycondensation.
6 samples, 1 request(s)

=== Request 1/1 ===

All values (4; kept 4):


,kept,sample_id,sample,paper_label,identifying_data,match_basis,method,subtype,value_raw,value,value_type,unit,origin,conditions,description,check
0,True,1555566,PHF | PHF,PHF,"homopolymer, semicrystalline, Tg = 16.7 °C, Tc ≈ 84 °C",same_entry,DSC,peak,151.3 °C,151.3,exact,°C,measured,heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=−120 to 225 °C (range),"Melting temperature of the PHF homopolymer, taken as the peak maximum of the melting endotherm in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −120 to 225 °C in a heating-cooling-heating cycle on a Mettler Toledo DSC1.",
1,True,1555570,PHF-b-F-pTHF | PHF-b-F-pTHF 35/65,PHF-b-F-pTHF 35/65,"35/65 weight ratio of PHF to F-pTHF segments, highest content of pTHF",same_entry,DSC,peak,−7.1 °C,-7.1,exact,°C,measured,heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=−120 to 225 °C (range),"Melting temperature of pTHF crystallites in the PHF-b-F-pTHF 35/65 copolymer, taken as the peak maximum of the melting endotherm in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −120 to 225 °C in a heating-cooling-heating cycle on a Mettler Toledo DSC1.",
2,True,1555571,PHF-b-F-pTHF | PHF-b-F-pTHF 25/75,PHF-b-F-pTHF 25/75,"25/75 weight ratio of PHF to F-pTHF segments, highest content of pTHF",same_entry,DSC,peak,−2.1 °C,-2.1,exact,°C,measured,heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=−120 to 225 °C (range),"Melting temperature of pTHF crystallites in the PHF-b-F-pTHF 25/75 copolymer, taken as the peak maximum of the melting endotherm in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −120 to 225 °C in a heating-cooling-heating cycle on a Mettler Toledo DSC1.",
3,True,1555571,PHF-b-F-pTHF | PHF-b-F-pTHF 25/75,PHF-b-F-pTHF 25/75,"25/75 weight ratio of PHF to F-pTHF segments, investigated as thermoplastic shape memory polymer",same_entry,DSC,peak,−2.8 °C,-2.8,exact,°C,measured,heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=−120 to 225 °C (range),"Low-temperature melting temperature of pTHF crystallites in the PHF-b-F-pTHF 25/75 copolymer, taken as the peak maximum of the melting endotherm in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −120 to 225 °C in a heating-cooling-heating cycle on a Mettler Toledo DSC1; used as the switching transition temperature for shape memory behavior.",



Kept values per sample:


,sample_id,name,identifier,kept
0,1555566,PHF,PHF,151.3 °C
1,1555567,PHF-b-F-pTHF,PHF-b-F-pTHF 75/25,
2,1555568,PHF-b-F-pTHF,PHF-b-F-pTHF 65/35,
3,1555569,PHF-b-F-pTHF,PHF-b-F-pTHF 50/50,
4,1555570,PHF-b-F-pTHF,PHF-b-F-pTHF 35/65,−7.1 °C
5,1555571,PHF-b-F-pTHF,PHF-b-F-pTHF 25/75,−2.1 °C; −2.8 °C


## Send prompt 3 (per sample, other samples as context)

One request per sample in `run_ids`. Records are in `p3_records`; reasoning in `p3_reasoning`.

In [265]:
p3_property = PROPERTY
p3_system = build_system_prompt(PROPERTY, "with_context")
p3_user = build_user_prompt(PROPERTY, "with_context")
p3_format = make_response_format(PROPERTY)


def build_p3_messages(sample_id):
    target = paper_samples.loc[paper_samples.sample_id == sample_id]
    if len(target) != 1:
        raise ValueError(f"Sample {sample_id} does not belong to paper {PAPER_UID}.")
    t = target.iloc[0]
    others = paper_samples.loc[paper_samples.sample_id != sample_id]
    values = {
        "paper_uid": PAPER_UID,
        "sample_id": str(sample_id),
        "sample_record": json.dumps({k: t[k] for k in ("name", "identifier", "aliases", "description")},
                                    ensure_ascii=False),
        "other_samples": "\n".join(sample_json(r) for r in others.itertuples()) or "[none]",
        "markdown_full_text": paper_markdown,
    }
    user = re.sub(r"\{(paper_uid|sample_id|sample_record|other_samples|markdown_full_text)\}",
                  lambda m: values[m.group(1)], p3_user)
    return [{"role": "system", "content": p3_system}, {"role": "user", "content": user}]


semaphore = asyncio.Semaphore(MAX_CONCURRENT)
async with AsyncOpenAI(base_url=BASE_URL, api_key="none", timeout=1200, max_retries=0) as client:
    async def run_one(sample_id):
        async with semaphore:
            return await ask_qwen(client, build_p3_messages(sample_id), p3_format)
    p3_responses = dict(zip(run_ids, await asyncio.gather(*(run_one(i) for i in run_ids),
                                                          return_exceptions=True)))
p3_reasoning = {i: get_reasoning(r) for i, r in p3_responses.items()}

rows = []
print(f"Property: {PROPERTY}")
for sample_id, response in p3_responses.items():
    print(f"\n=== {sample_id}: {names['name'][sample_id]} | {names['identifier'][sample_id]} ===")
    print_reasoning(response)
    parsed, note = parse_response(response)
    records = parsed.get("properties") if isinstance(parsed, dict) else None
    if records is None:
        print(note or "[No parsable answer]")
        continue
    if not records:
        print("No values reported for this sample.")
    for r in records:
        rows.append({"sample_id": sample_id, **record_row(r), "check": check_record(r, PROPERTY)})
    if note:
        print(note)

p3_records = pd.DataFrame(rows)
if not p3_records.empty:
    dup = p3_records.groupby(["value_raw", "method", "conditions"]).sample_id.transform("nunique") > 1
    p3_records.loc[dup, "check"] = (p3_records.loc[dup, "check"] + "; assigned to several samples").str.strip("; ")
    print(f"\nPrompt 3 values ({len(p3_records)}):")
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(p3_records)

Property: melting_temperature

=== 1555566: PHF | PHF ===

=== 1555567: PHF-b-F-pTHF | PHF-b-F-pTHF 75/25 ===
No values reported for this sample.

=== 1555568: PHF-b-F-pTHF | PHF-b-F-pTHF 65/35 ===
No values reported for this sample.

=== 1555569: PHF-b-F-pTHF | PHF-b-F-pTHF 50/50 ===
No values reported for this sample.

=== 1555570: PHF-b-F-pTHF | PHF-b-F-pTHF 35/65 ===

=== 1555571: PHF-b-F-pTHF | PHF-b-F-pTHF 25/75 ===

Prompt 3 values (4):


,sample_id,method,subtype,value_raw,value,value_type,unit,origin,conditions,description,check
0,1555566,DSC,peak,151.3 °C,151.3,exact,°C,measured,heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=-120 to 225 °C (range),"Melting temperature of the semicrystalline polyester, taken as the maximum of the melting endotherm in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −120 to 225 °C in a heating–cooling–heating cycle on a Mettler Toledo DSC1 calibrated with indium and n-octane.",
1,1555570,DSC,peak,−7.1 °C,-7.1,exact,°C,measured,heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=-120 to 225 °C (range),"Melting temperature of pTHF crystallites, taken as the peak maximum of the melting endotherm in the second DSC heating scan, recorded at 10 °C/min under nitrogen from −120 to 225 °C in a heating–cooling–heating cycle; no specimen form stated.",
2,1555571,DSC,peak,−2.1 °C,-2.1,exact,°C,measured,heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=nitrogen; temperature_range=-120 to 225 °C (range),"Melting temperature of the pTHF crystalline phase, taken as the peak maximum of the melting endotherm in the second heating scan of a DSC heating–cooling–heating cycle on a Mettler Toledo DSC1, recorded at 10 °C/min under nitrogen from −120 to 225 °C after cooling at 10 °C/min; no specimen form or mass stated.",
3,1555571,DSC,peak,−2.8 °C,-2.8,exact,°C,measured,heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=nitrogen; temperature_range=-120 to 225 °C (range),"Low-temperature melting temperature of the pTHF crystalline phase, taken as the peak maximum of the DSC melting endotherm in the second heating scan of a heating–cooling–heating cycle on a Mettler Toledo DSC1, recorded at 10 °C/min under nitrogen from −120 to 225 °C after cooling at 10 °C/min; reported as the switching transition temperature for shape memory behavior; no specimen form or mass stated.",


## Compare the three prompts

One row per sample that prompts 1 and 3 were run for. `p2_kept` are the values prompt 2 kept for that sample; `p2_weak` are values prompt 2 linked to the sample but did not keep (`name_only` with a shared name). `same` is True when all three prompts return the same set of values (by `value_raw` and method). Values prompt 2 linked to no sample are listed below the table. A warning is printed if the prompts were run for different properties.

In [266]:
def labels(df, mask=None):
    # "value_raw [method/subtype; conditions]" per sample_id.
    if df is None or df.empty:
        return {}
    if mask is not None:
        df = df[mask]
    out = {}
    for r in df.to_dict("records"):
        method = r.get("method") or ""
        if r.get("subtype") not in (None, "unspecified"):
            method += f"/{r['subtype']}"
        cond = r.get("conditions") or ""
        out.setdefault(r["sample_id"], []).append(f"{r.get('value_raw')} [{method}{'; ' + cond if cond else ''}]")
    return out


def value_keys(df, sample_id, mask=None):
    if df is None or df.empty:
        return set()
    d = df if mask is None else df[mask]
    d = d[d.sample_id == sample_id]
    return {(str(v).strip(), m) for v, m in zip(d.value_raw, d.method)}


run_props = {name: globals().get(f"{name}_property") for name in ("p1", "p2", "p3")}
if len({p for p in run_props.values() if p}) > 1:
    print(f"WARNING: the prompts were run for different properties: {run_props}")

p1 = globals().get("p1_records", pd.DataFrame())
p2 = globals().get("p2_records", pd.DataFrame())
p3 = globals().get("p3_records", pd.DataFrame())
p2_kept_mask = p2.kept if not p2.empty else None
p2_weak_mask = (~p2.kept & p2.sample_id.notna()) if not p2.empty else None

p1_map, p3_map = labels(p1), labels(p3)
p2_kept_map, p2_weak_map = labels(p2, p2_kept_mask), labels(p2, p2_weak_mask)

rows = []
for sample_id in run_ids:
    r = names.loc[sample_id]
    k1 = value_keys(p1, sample_id)
    k2 = value_keys(p2, sample_id, p2_kept_mask)
    k3 = value_keys(p3, sample_id)
    rows.append({
        "sample_id": sample_id,
        "sample": f"{r['name']} | {r['identifier']}",
        "p1_per_sample": " | ".join(p1_map.get(sample_id, [])),
        "p2_kept": " | ".join(p2_kept_map.get(sample_id, [])),
        "p2_weak": " | ".join(p2_weak_map.get(sample_id, [])),
        "p3_with_context": " | ".join(p3_map.get(sample_id, [])),
        "same": k1 == k2 == k3,
    })
comparison = pd.DataFrame(rows)
print(f"Property: {PROPERTY}")
print(f"{PAPER_UID}: {get_paper_title(PAPER_UID)}")
print(f"Samples compared: {len(comparison)}; all three agree on {int(comparison.same.sum())}")
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(comparison)

if not p2.empty:
    unmatched = p2[p2.sample_id.isna()]
    print(f"\nPrompt 2 values linked to no sample ({len(unmatched)}):")
    if len(unmatched):
        with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
            display(unmatched[["paper_label", "identifying_data", "value_raw", "method", "subtype", "conditions"]])
    others = p2[p2.kept & ~p2.sample_id.isin(run_ids)]
    if len(others) and not RUN_ALL_SAMPLES:
        print(f"\nPrompt 2 also kept {len(others)} value(s) for samples not run with prompts 1 and 3 "
              f"(set RUN_ALL_SAMPLES = True to compare them).")

Property: melting_temperature
ID001032337: Biobased Thermoplastic Elastomers: Structure-Property Relationship of Poly(hexamethylene 2,5-furanodicarboxylate)-Block-Poly(tetrahydrofuran) Copolymers Prepared by Melt Polycondensation.
Samples compared: 6; all three agree on 6


,sample_id,sample,p1_per_sample,p2_kept,p2_weak,p3_with_context,same
0,1555566,PHF | PHF,151.3 °C [DSC/peak; scan=second_heating; heating_rate=10 °C/min; cooling_rate=10 °C/min; atmosphere=N2; temperature_range=-120 to 225 °C (range)],151.3 °C [DSC/peak; heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=−120 to 225 °C (range)],,151.3 °C [DSC/peak; heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=-120 to 225 °C (range)],True
1,1555567,PHF-b-F-pTHF | PHF-b-F-pTHF 75/25,,,,,True
2,1555568,PHF-b-F-pTHF | PHF-b-F-pTHF 65/35,,,,,True
3,1555569,PHF-b-F-pTHF | PHF-b-F-pTHF 50/50,,,,,True
4,1555570,PHF-b-F-pTHF | PHF-b-F-pTHF 35/65,−7.1 °C [DSC/peak; scan=second_heating; heating_rate=10 °C/min; cooling_rate=10 °C/min; atmosphere=N2; temperature_range=-120 to 225 °C (range)],−7.1 °C [DSC/peak; heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=−120 to 225 °C (range)],,−7.1 °C [DSC/peak; heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=-120 to 225 °C (range)],True
5,1555571,PHF-b-F-pTHF | PHF-b-F-pTHF 25/75,−2.1 °C [DSC/peak; heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=−120 to 225 °C (range)] | −2.8 °C [DSC/peak; heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=−120 to 225 °C (range)],−2.1 °C [DSC/peak; heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=−120 to 225 °C (range)] | −2.8 °C [DSC/peak; heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=N2; temperature_range=−120 to 225 °C (range)],,−2.1 °C [DSC/peak; heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=nitrogen; temperature_range=-120 to 225 °C (range)] | −2.8 °C [DSC/peak; heating_rate=10 °C/min; cooling_rate=10 °C/min; scan=second_heating; atmosphere=nitrogen; temperature_range=-120 to 225 °C (range)],True



Prompt 2 values linked to no sample (0):


## All records with all fields

One cell per prompt. Each prints every record that prompt returned, as JSON with all fields: grouped by sample for prompts 1 and 3, and by request for prompt 2, where each record also shows the sample it was linked to and whether code kept it. Run the matching send cell first.

In [267]:
# Prompt 1 (per sample): every record with all fields.
print("=" * 80)
print(f"PROMPT 1 (per sample) - {PAPER_UID} - {globals().get('p1_property')}")
print("=" * 80)
if "p1_responses" not in globals():
    print("Not run.")
else:
    for sample_id, response in p1_responses.items():
        records, note = records_of(response)
        print_records(sample_title(sample_id), records or [])
        if records is None or note:
            print(note or "[No parsable answer]")

PROMPT 1 (per sample) - ID001032337 - melting_temperature

--- 1555566: PHF | PHF: 1 record(s) ---

[1]
{
  "property": "melting_temperature",
  "method": "DSC",
  "subtype": "peak",
  "value_raw": "151.3 °C",
  "value": 151.3,
  "value_type": "exact",
  "unit": "°C",
  "conditions": [
    {
      "name": "scan",
      "type": "exact",
      "value": "second_heating",
      "unit": null,
      "description": null
    },
    {
      "name": "heating_rate",
      "type": "exact",
      "value": 10,
      "unit": "°C/min",
      "description": null
    },
    {
      "name": "cooling_rate",
      "type": "exact",
      "value": 10,
      "unit": "°C/min",
      "description": null
    },
    {
      "name": "atmosphere",
      "type": "exact",
      "value": "N2",
      "unit": null,
      "description": null
    },
    {
      "name": "temperature_range",
      "type": "range",
      "value": "-120 to 225",
      "unit": "°C",
      "description": null
    }
  ],
  "origin": "measured",


In [268]:
# Prompt 2 (per paper): the same layout as prompts 1 and 3. One block per sample, in the paper's
# sample order, each record printed as the same property record (same fields, same order).
# Under each record: how prompt 2 linked it and whether code kept it. Values linked to no sample come last.
print("=" * 80)
print(f"PROMPT 2 (per paper) - {PAPER_UID} - {globals().get('p2_property')}")
print("=" * 80)

if "p2_responses" not in globals():
    print("Not run.")
else:
    by_sample, unlinked, seen_unlinked = {}, [], set()
    for b, (targets, response) in enumerate(zip(batches, p2_responses), 1):
        parsed_records, note = records_of(response)
        if parsed_records is None or note:
            print(f"Request {b}/{len(batches)}: {note or '[No parsable answer]'}")
        for r in parsed_records or []:
            sid = r.get("sample_id")
            if sid is None:
                key = (r.get("paper_label"), r.get("value_raw"), r.get("method"))
                if key not in seen_unlinked:  # the same unlinked value can come back from every batch
                    seen_unlinked.add(key)
                    unlinked.append(r)
            elif sid in targets:              # a link only counts from the request that targeted it
                by_sample.setdefault(sid, []).append(r)

    def link_line(r):
        status = "kept" if p2_kept(r) else "not kept"
        return (f"    match_basis: {r.get('match_basis')} ({status}) | paper_label: {r.get('paper_label')!r} | "
                f"identifying_data: {r.get('identifying_data')!r}")

    def print_block(title, records):
        print(f"\n--- {title}: {len(records)} record(s) ---")
        if not records:
            print("(no records)")
        for n, r in enumerate(records, 1):
            print(f"\n[{n}]")
            print(json.dumps({k: r.get(k) for k in RECORD_FIELDS}, ensure_ascii=False, indent=2))
            print(link_line(r))

    for sample_id in [int(i) for i in paper_samples.sample_id]:
        print_block(sample_title(sample_id), by_sample.get(sample_id, []))
    print_block("No sample (values linked to no listed sample)", unlinked)

PROMPT 2 (per paper) - ID001032337 - melting_temperature

--- 1555566: PHF | PHF: 1 record(s) ---

[1]
{
  "property": "melting_temperature",
  "method": "DSC",
  "subtype": "peak",
  "value_raw": "151.3 °C",
  "value": 151.3,
  "value_type": "exact",
  "unit": "°C",
  "conditions": [
    {
      "name": "heating_rate",
      "type": "exact",
      "value": 10,
      "unit": "°C/min",
      "description": null
    },
    {
      "name": "cooling_rate",
      "type": "exact",
      "value": 10,
      "unit": "°C/min",
      "description": null
    },
    {
      "name": "scan",
      "type": "exact",
      "value": "second_heating",
      "unit": null,
      "description": null
    },
    {
      "name": "atmosphere",
      "type": "exact",
      "value": "N2",
      "unit": null,
      "description": null
    },
    {
      "name": "temperature_range",
      "type": "range",
      "value": "−120 to 225",
      "unit": "°C",
      "description": null
    }
  ],
  "origin": "measured",
 

In [269]:
# Prompt 3 (per sample, other samples as context): every record with all fields.
print("=" * 80)
print(f"PROMPT 3 (per sample, other samples as context) - {PAPER_UID} - {globals().get('p3_property')}")
print("=" * 80)
if "p3_responses" not in globals():
    print("Not run.")
else:
    for sample_id, response in p3_responses.items():
        records, note = records_of(response)
        print_records(sample_title(sample_id), records or [])
        if records is None or note:
            print(note or "[No parsable answer]")

PROMPT 3 (per sample, other samples as context) - ID001032337 - melting_temperature

--- 1555566: PHF | PHF: 1 record(s) ---

[1]
{
  "property": "melting_temperature",
  "method": "DSC",
  "subtype": "peak",
  "value_raw": "151.3 °C",
  "value": 151.3,
  "value_type": "exact",
  "unit": "°C",
  "conditions": [
    {
      "name": "heating_rate",
      "type": "exact",
      "value": 10,
      "unit": "°C/min",
      "description": null
    },
    {
      "name": "cooling_rate",
      "type": "exact",
      "value": 10,
      "unit": "°C/min",
      "description": null
    },
    {
      "name": "scan",
      "type": "exact",
      "value": "second_heating",
      "unit": null,
      "description": null
    },
    {
      "name": "atmosphere",
      "type": "exact",
      "value": "N2",
      "unit": null,
      "description": null
    },
    {
      "name": "temperature_range",
      "type": "range",
      "value": "-120 to 225",
      "unit": "°C",
      "description": null
    }
  ]